In [1]:
from qdrant_client import models, QdrantClient
print(1)
from qdrant_client.models import SparseVector
print(2)
from fastembed import SparseTextEmbedding
print(3)


s:\my_projects\Arabic_News_Agentic_RAG\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1
2
3


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
print(4)

4


In [3]:
from sentence_transformers import SentenceTransformer
print("import ok")

import ok


In [4]:
import torch
print(torch.__version__)

2.7.1+cu118


In [5]:
import tokenizers
print(tokenizers.__version__)

0.22.2


In [6]:
#!pip install -U torch torchvision sentence-transformers tokenizers sentencepiece

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
print("Model loaded")

vec = model.encode("ما هي آخر الأخبار السياسية؟", normalize_embeddings=True)
print(f"Dimension: {len(vec)}")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("aubmindlab/bert-base-arabertv02", device="cpu")
print("Model loaded")

c:\Users\saleen\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("aubmindlab/bert-base-arabertv02", device="cpu")
print("Model loaded")

# Test it
vec = model.encode("ما هي آخر الأخبار السياسية؟", normalize_embeddings=True)
print(f"Dimension: {len(vec)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1008.44it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Dimension: 768


In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
    )

print("Dense model loaded")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2150.85it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense model loaded


In [9]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
print("Sparse model loaded")


client = QdrantClient(path="./data/qdrant_db")

collection_name = "arabic_news"

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 10.31it/s]


Sparse model loaded


C:\Users\saleen\AppData\Local\Temp\ipykernel_19112\3039890167.py:5: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <arabic_news> contains 30148 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="./data/qdrant_db")


In [ ]:
def retrieve(query, top_k=5):
    dense_vec = embeddings.embed_query(query)

    sparse_vec = list(sparse_model.embed([query]))[0]

    results = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=dense_vec,
                using="",
                limit=20
            ),
            models.Prefetch(
                query=SparseVector(
                    indices=sparse_vec.indices.tolist(),
                    values=sparse_vec.values.tolist()
                ),
                using="sparse",
                limit=20
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    )
    return results.points

In [11]:
queries = [
    "ما هي آخر الأخبار السياسية؟",      # politics
    "نتائج كأس العالم لكرة القدم",        # sports
    "أخبار الاقتصاد والأسواق المالية"     # finance
]

for q in queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, top_k=3)
    for r in results:
        print(f"  [{r.payload['category']}] {r.payload['text'][:100]}...")



Query: ما هي آخر الأخبار السياسية؟
  [Politics] ، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابلة التي تجريها الزميلة ريما مكتبي مع جنبلاط الفراغ ...
  [Politics] . ولعل ابرز الشعارات او الاهداف التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يح...
  [Politics] جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشتراكي اللبناني وليد جنبلاط غدا ا...

Query: نتائج كأس العالم لكرة القدم
  [Politics] رئيس الاتحاد الالماني يستبعد اقامة كاس العالم في قطر الحدث.نت استبعد رئيس الاتحاد الالماني لكرة القد...
  [Medical] تحذير.. ضربة الكرة بالراس قد تكون قاتلة العربية.نت عماد البليك دعا اتحاد كرة القدم الانجليزي، الجمعة...
  [Politics] ، الليلة عن تفاصيل بنود الوثيقة، وستعلن عن نتائج الحركة وانتخاباتها الداخلية في الايام المقبلة. اسرا...

Query: أخبار الاقتصاد والأسواق المالية
  [Politics] ، الساعة الحادية عشر مساء بتوقيت المملكة العربية السعودية...
  [Politics] ، بعد ان خابت توقعاته بنمو في الاقتصاد للعام ، يعوض خسائر العقوبات. ويجبره التورط في سورية، تارة على

In [12]:
print(client.get_collection("arabic_news"))


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=30148 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors={'sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=False, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshol

In [16]:
retrieve("ماذا يحدث في مصر؟", top_k=3)

[ScoredPoint(id='43583cb6-f71a-4a9a-8a2c-ede303890904', version=0, score=0.5, payload={'text': 'تونس.. قتيلا بهجوم انتحاري والسبسي يعلن الطوارئ لا يوجد', 'category': 'Politics', 'source_idx': 2212}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='ad90e063-bedd-4599-b8f1-caf358079930', version=0, score=0.5, payload={'text': '، على سؤال ماذا يحدث في دماغ شاب تعرض لوابل من الافلام الاباحية قائلا تتغير الخلايا الدماغية في ظل اكتساب المعارف. ويضر التعلم في حالة من الادمان بالدماغ كثيرا، فنصبح متمسكين ببعض انماط السلوك وبعض الاذواق. وعندما يكون الدماغ بانتظار مكافاة ما، كما هي الحال في الافلامالاباحية', 'category': 'Medical', 'source_idx': 3923}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='01a21d50-3b8f-4bd3-b6d7-b5cbb2353c45', version=0, score=0.39215686274509803, payload={'text': 'في مصر.. تتغير الحكومات وتبقى مشكلة المرور الحدث.نت كل شيء يمكن ان يتغير في مصر.. الحكومة او الرئيس، ولكن تبقى مشكلة المرور عصية على التغيير', 'category': 'Politics', 'source